# Part 6 - Conclusions and reproducibility

BFT65B final report, walkthrough notebook. Every number below is read from `reports/` or recomputed live by the same scripts the README lists. Seed: `config.SEED = 42`.

In [1]:
import json, os, subprocess, sys
from pathlib import Path

import pandas as pd
import polars as pl
from IPython.display import Image, Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import config as C

pd.set_option("display.max_columns", 30, "display.width", 200, "display.max_colwidth", 80)
R = ROOT / "reports"
print("project root:", ROOT, "| python", sys.version.split()[0], "| polars", pl.__version__, "| seed", C.SEED)


def run(script, *args, tail=25):
    """Run one pipeline script exactly as the README does, and show the end of its output."""
    t = subprocess.run([sys.executable, script, *args], cwd=ROOT, capture_output=True, text=True,
                       encoding="utf-8", errors="replace", env={**os.environ, "PYTHONUTF8": "1"})
    lines = (t.stdout + t.stderr).strip().splitlines()
    print("\n".join(lines[-tail:]))
    print(f"\n>>> {script} {' '.join(args)} exited with {t.returncode}")
    assert t.returncode == 0, "step failed"


def csv(name, n=None):
    df = pd.read_csv(R / name)
    display(df if n is None else df.head(n))


def fig(name, width=820):
    display(Image(filename=str(R / name), width=width))

project root: D:\Final BigData | python 3.15.2 | polars 2.1.0 | seed 42


**Goal.** Answer Q1-Q3 from the measured results, state the limitations, and show that the pipeline reruns.
`p6_conclusions.py` stops if any test suite has a failure.

In [2]:
run('src/p6_conclusions.py')

   -0.04169570347277734,
   0.08617757595455545
  ],
  "bank_recall_actionable": 0.7564102564102564,
  "caught": 59,
  "late": 9,
  "no_alarm": 10,
  "precision": 0.07225630538513975,
  "alarm_share": 0.010706778770362585,
  "saving_pct": 2.077807415627656,
  "saving_pct_without_two": 18.114565077733943,
  "no_system_usd_bn": 38.67633414999999,
  "with_system_usd_bn": 37.87271441093836
 },
 "spark_vs_sklearn": {
  "ap_spark": 0.59116,
  "ap_sklearn": 0.58949,
  "rank_corr": 0.9938734277276113
 },
 "tests_passed": {
  "part2": 20,
  "part4": 31,
  "part5": 16
 }
}

>>> src/p6_conclusions.py  exited with 0


In [3]:
print(json.dumps(json.loads((R / 'p6_key_numbers.json').read_text()), indent=1))

{
 "storage_mb": {
  "csv": 490.2,
  "chosen": 206.3
 },
 "engines_s": {
  "polars": 1.785,
  "pandas": 14.526,
  "spark_18": 77.742,
  "spark_200": 105.963
 },
 "scale": {
  "spark_max_scale_finished": 16,
  "spark_max_s": 345.995,
  "spark_max_spill_gb": 14.394,
  "spark_failed_scales": [],
  "polars_x2_s": 4.191,
  "polars_x4": "memory_guard",
  "pandas_x4": "memory_guard"
 },
 "split": {
  "dev": {
   "rows": 1346634,
   "failed_banks": 2755
  },
  "gap": {
   "rows": 28805,
   "failed_banks": 59
  },
  "test": {
   "rows": 274032,
   "failed_banks": 78
  },
  "unlabelled": {
   "rows": 17248,
   "failed_banks": 0
  }
 },
 "features": {
  "candidates": 42,
  "selected": 25
 },
 "cv_mean_ap": {
  "chosen": 0.4531,
  "B1": 0.3322,
  "B0": 0.296
 },
 "test": {
  "ap": 0.34547,
  "ap_ci": [
   0.2509837737271677,
   0.4285084116559973
  ],
  "ap_B1": 0.31774,
  "ap_B0": 0.09587,
  "diff_vs_B1_ci": [
   -0.04169570347277734,
   0.08617757595455545
  ],
  "bank_recall_actionable": 0.7564

In [4]:
display(Markdown((R / 'p6_first_page.md').read_text(encoding='utf-8')))

# First page facts (generated, fill in the name, ID and URL)

| Item | Value |
|---|---|
| Name, student ID | _to fill in_ |
| GitHub URL | _to fill in_ |
| Download method | FDIC BankFind API, `fdic_download_full.py` (resumable) |
| Download date | 2026-11-07 (16.5 minutes) |
| Financials | 1,682,616 rows, 24,064 banks, 170 quarters, 1984Q1-2026Q2 |
| Failures, Institutions | 4,117 and 27,835 rows |
| Clean panel | 1,666,719 bank-quarters |
| Development | 1984Q1-2011Q4: 1,346,634 rows, 2,755 failed banks labelled |
| Gap | 2012Q1-2012Q4, not used |
| Test | 2013Q1-2025Q2: 274,032 rows, 78 failed banks labelled |


### Clean rerun: exit codes and time of every step

In [5]:
csv('repro_status.csv')

,step,exit,seconds
0,src/p1_verify.py,0.0,9.0
1,src/p1_build_bronze.py,0.0,4.0
2,src/p1_storage_benchmark.py --repeat 1,0.0,250.0
3,src/p2_build_panel.py,0.0,2.0
4,src/p2_quality.py,0.0,14.0
5,src/p2_checks.py,0.0,14.0
6,src/p3_benchmark.py --suite engines --repeat 2 --spark-repeat 1,0.0,458.0
7,src/p3_compare.py,0.0,13.0
8,src/p3_benchmark.py --suite threads --repeat 1 --spark-repeat 1,0.0,700.0
9,src/p3_benchmark.py --suite spark_tune --spark-repeat 1,0.0,646.0


In [6]:
df = pd.read_csv(R / 'repro_check.csv'); display(df.groupby(['kind', 'same']).size().rename('files').reset_index()); df.head(15)

,kind,same,files
0,deterministic,True,61
1,variable,False,24
2,variable,True,5


,file,kind,same,detail
0,p1_bench_env.json,variable,True,bytes
1,p1_bronze_build.json,variable,False,/financials/seconds
2,p1_completeness.json,variable,False,/raw_folder
3,p1_control_totals.csv,variable,True,bytes
4,p1_failures_coverage.csv,deterministic,True,bytes
5,p1_parquet_columns.csv,deterministic,True,bytes
6,p1_parquet_row_groups.json,deterministic,True,bytes
7,p1_polars_plans.txt,variable,False,text differs
8,p1_raw_sha256.csv,deterministic,True,bytes
9,p1_schema_drift.csv,deterministic,True,bytes
